In [5]:
# Células 1-6 (Mantidas como no original)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import joblib
import json
import os # Importar OS para manipulação de arquivos

# Carregar o DataFrame
df_matrix = pd.read_csv(r"C:\Users\lucas.galicioli\Desktop\ifc-classifier\notebooks\data\interim\cls_matrix_v2.csv") # Use seu caminho

# Limpeza inicial (como na sua Célula 5 original)
colunas_para_limpar = [
    'Material',
    'PSET_RÔGGA.RÔGGA_SEÇÃO',
    'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
    'Ô_CLS_DISCIPLINAS',
    'Ô_CLS_CLASSIFICAÇÃO_SOLIBRI'
]
valor_ausente = 'NaN' # Assumindo que 'NaN' é uma string lida do CSV
novo_valor = 'Desconhecido'
for coluna in colunas_para_limpar:
    if coluna in df_matrix.columns:
         # Preenche NaNs reais e a string 'NaN'
        df_matrix[coluna] = df_matrix[coluna].fillna(novo_valor)
        df_matrix[coluna] = df_matrix[coluna].replace(valor_ausente, novo_valor)

print("DataFrame carregado e limpeza inicial concluída.")
print(f"Total de linhas: {len(df_matrix)}")

DataFrame carregado e limpeza inicial concluída.
Total de linhas: 311763


C:\Users\lucas.galicioli\AppData\Local\Temp\ipykernel_10176\1591954481.py:14: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_matrix = pd.read_csv(r"C:\Users\lucas.galicioli\Desktop\ifc-classifier\notebooks\data\interim\cls_matrix_v2.csv") # Use seu caminho


In [6]:
# CÉLULA DO LOOP PRINCIPAL - VERSÃO COM TF-IDF NO 'Name'
from sklearn.feature_extraction.text import TfidfVectorizer # <-- Importar

# --- Configurações ---
TARGET_COLUMN = 'Ô_CLS_CLASSIFICAÇÃO_SOLIBRI'
DISCIPLINE_COLUMN = 'Ô_CLS_DISCIPLINAS'
MIN_SAMPLES_PER_CLASS = 3 
TEST_SIZE = 0.25
RANDOM_STATE = 42
OUTPUT_DIR = "modelos_por_disciplina_tfidf" # <-- Nova pasta de saída

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Configurações do TF-IDF ---
# max_features: Limita o número de colunas criadas pelo TF-IDF (controla dimensionalidade)
# ngram_range: (1, 2) considera palavras isoladas e pares de palavras (ex: "caixa passagem")
TFIDF_MAX_FEATURES = 100 # <-- Ajustável: Comece com 100, pode aumentar/diminuir
TFIDF_NGRAM_RANGE = (1, 2)

# --- Identificar Disciplinas ---
disciplinas = df_matrix[DISCIPLINE_COLUMN].unique()
print(f"Disciplinas encontradas: {disciplinas}")

# --- Dicionário para armazenar os TF-IDF Vectorizers treinados ---
tfidf_vectorizers = {} # <-- Armazenaremos um por disciplina

# --- Loop Principal por Disciplina ---
for discipline in disciplinas:
    print(f"\n{'='*30} PROCESSANDO DISCIPLINA: {discipline} {'='*30}")

    df_discipline = df_matrix[df_matrix[DISCIPLINE_COLUMN] == discipline].copy()
    print(f"-> Número de amostras para '{discipline}': {len(df_discipline)}")
    if len(df_discipline) < 10: 
        print(f"AVISO: Pulando disciplina '{discipline}'.")
        continue

    # 2. Definir Features (X) e Target (y)
    features_selecionadas = [
        'Class', 'PredefinedType', 'BuildingStorey',
        'Material', 'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO',
        'Name' # <-- Incluir 'Name' para TF-IDF
    ]
    features_presentes = [col for col in features_selecionadas if col in df_discipline.columns]
    X_disc = df_discipline[features_presentes].copy()
    y1_disc_original = df_discipline[TARGET_COLUMN]
    print(f"-> Features iniciais selecionadas: {features_presentes}")

    # 3. Pré-processamento de X 
    
    # 3.1 Tratar NaNs Categóricos (Incluindo 'Name' agora)
    placeholder = "Desconhecido"
    cat_cols_all = X_disc.select_dtypes(include=['object']).columns
    X_disc.loc[:, cat_cols_all] = X_disc.loc[:, cat_cols_all].fillna(placeholder)

    # 3.2 <-- MUDANÇA: Aplicar TF-IDF no 'Name'
    print("-> Aplicando TF-IDF no 'Name'...")
    tfidf = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=TFIDF_NGRAM_RANGE)
    
    # Fit_transform nos dados COMPLETOS da disciplina ANTES do split
    tfidf_features = tfidf.fit_transform(X_disc['Name']) 
    
    # Salvar o vectorizer treinado para usar na aplicação
    tfidf_vectorizers[discipline] = tfidf 
    
    # Criar um DataFrame com as features TF-IDF
    df_tfidf = pd.DataFrame(tfidf_features.toarray(), 
                            columns=[f"Name_tfidf_{i}" for i in range(tfidf_features.shape[1])],
                            index=X_disc.index) # <-- Manter o índice original
                            
    # Remover a coluna 'Name' original
    X_disc = X_disc.drop('Name', axis=1)
    
    # Concatenar as features TF-IDF com o restante de X_disc
    X_disc = pd.concat([X_disc, df_tfidf], axis=1)
    print(f"   Coluna 'Name' substituída por {df_tfidf.shape[1]} features TF-IDF.")
    print(f"   Shape atual de X: {X_disc.shape}")
    
    # 3.3 Aplicar Frequency Encoding (Nas colunas restantes)
    colunas_para_freq_encoding = [
        'Material', 'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO'
    ]
    print("-> Aplicando Frequency Encoding...")
    for col in colunas_para_freq_encoding:
        if col in X_disc.columns:
            frequencias = X_disc[col].value_counts(normalize=True)
            X_disc[col + '_Freq'] = X_disc[col].map(frequencias)
            X_disc[col + '_Freq'] = X_disc[col + '_Freq'].fillna(0) 
            X_disc = X_disc.drop(col, axis=1)
            
    # 3.4 Aplicar One-Hot Encoding (Nas colunas restantes)
    print("-> Aplicando One-Hot Encoding...")
    X_encoded_disc = pd.get_dummies(X_disc) 
    X_encoded_disc.columns = X_encoded_disc.columns.str.replace(r'[\[\]<]', '_', regex=True)
    print(f"   Shape após OHE: {X_encoded_disc.shape}")
    
    colunas_modelo_disciplina = X_encoded_disc.columns.tolist()

    # 4. Pré-processamento de y (Filtragem, Encoders) - SEM MUDANÇAS
    # (Código omitido para brevidade - mantenha como estava)
    le1_disc = LabelEncoder()
    y1_encoded_disc = le1_disc.fit_transform(y1_disc_original)
    print(f"-> Filtrando classes com menos de {MIN_SAMPLES_PER_CLASS} amostras...")
    classes_disc, counts_disc = np.unique(y1_encoded_disc, return_counts=True)
    valid_classes_disc = classes_disc[counts_disc >= MIN_SAMPLES_PER_CLASS]
    if len(valid_classes_disc) <= 1: continue
    mask_disc = np.isin(y1_encoded_disc, valid_classes_disc)
    X_filtered_disc = X_encoded_disc[mask_disc]
    y1_filtered_disc = y1_encoded_disc[mask_disc]
    print(f"   Amostras removidas: {len(y1_encoded_disc) - len(y1_filtered_disc)}")
    if X_filtered_disc.empty: continue
    le_xgb_disc = LabelEncoder()
    y_xgb_disc = le_xgb_disc.fit_transform(y1_filtered_disc)
    total_classes_disc = len(le_xgb_disc.classes_)
    print(f"-> Total de classes ÚNICAS para o modelo '{discipline}': {total_classes_disc}")

    # 5. Train/Test Split - SEM MUDANÇAS
    # (Código omitido para brevidade - mantenha como estava)
    print("-> Dividindo dados em treino/teste...")
    try:
        X_train_disc, X_test_disc, y_train_disc, y_test_disc = train_test_split(
            X_filtered_disc, y_xgb_disc, test_size=TEST_SIZE, 
            random_state=RANDOM_STATE, stratify=y_xgb_disc 
        )
    except ValueError as e: continue 
    print(f"   Shape X_train: {X_train_disc.shape}")

    # 6. Treinamento XGBoost (com Early Stopping e Pesos) - SEM MUDANÇAS
    #    (Mantendo sample_weight por enquanto)
    # (Código omitido para brevidade - mantenha como estava)
    print("-> Iniciando treinamento XGBoost com Early Stopping e Pesos...")
    xgb_classifier_disc = xgb.XGBClassifier( objective='multi:softmax', num_class=total_classes_disc, tree_method="hist", device="cuda", use_label_encoder=False, n_estimators=1000, learning_rate=0.05, max_depth=7, subsample=0.8, colsample_bytree=0.8, early_stopping_rounds=10, eval_metric='mlogloss', random_state=RANDOM_STATE )
    sample_weights_disc = compute_sample_weight(class_weight='balanced', y=y_train_disc)
    xgb_classifier_disc.fit( X_train_disc, y_train_disc, sample_weight=sample_weights_disc, eval_set=[(X_test_disc, y_test_disc)], verbose=False )
    print(f"   Treinamento concluído! Melhor iteração: {xgb_classifier_disc.best_iteration}")

    # 7. Avaliação - SEM MUDANÇAS
    # (Código omitido para brevidade - mantenha como estava)
    print("-> Avaliando o modelo...")
    y_pred_disc = xgb_classifier_disc.predict(X_test_disc)
    accuracy_disc = accuracy_score(y_test_disc, y_pred_disc)
    print(f"   Acurácia FINAL para '{discipline}': {accuracy_disc:.4f} ({accuracy_disc:.2%})")
    try:
        original_indices_test = le_xgb_disc.inverse_transform(y_test_disc)
        original_indices_pred = le_xgb_disc.inverse_transform(y_pred_disc)
        target_names_disc = le1_disc.inverse_transform(le_xgb_disc.classes_) 
        print(f"\n   Relatório de Classificação Detalhado para '{discipline}':")
        print(classification_report( original_indices_test, original_indices_pred, labels=le_xgb_disc.classes_, target_names=target_names_disc, zero_division=0 ))
    except Exception as e: print(f"   Erro ao gerar relatório: {e}")

    # 8. Salvar Artefatos - ADICIONAR TFIDF VECTORIZER
    print("-> Salvando artefatos...")
    discipline_code = str(discipline).replace(' ', '_').replace('/', '-') 
    # Salvar Modelo, Encoder, Colunas (como antes)
    model_path = os.path.join(OUTPUT_DIR, f"modelo_{discipline_code}.pkl")
    joblib.dump(xgb_classifier_disc, model_path)
    encoder_path = os.path.join(OUTPUT_DIR, f"encoder_le1_{discipline_code}.pkl")
    joblib.dump(le1_disc, encoder_path)
    columns_path = os.path.join(OUTPUT_DIR, f"colunas_{discipline_code}.json")
    with open(columns_path, 'w', encoding='utf-8') as f: json.dump(colunas_modelo_disciplina, f)
    
    # <-- MUDANÇA: Salvar o TFIDF Vectorizer treinado
    tfidf_path = os.path.join(OUTPUT_DIR, f"tfidf_{discipline_code}.pkl")
    joblib.dump(tfidf, tfidf_path) 
    
    print(f"   Modelo salvo: {model_path}")
    print(f"   Encoder salvo: {encoder_path}")
    print(f"   Colunas salvas: {columns_path}")
    print(f"   TFIDF Vectorizer salvo: {tfidf_path}") # <-- Nova linha

print(f"\n{'='*30} PROCESSAMENTO CONCLUÍDO {'='*30}")

# <-- MUDANÇA: Salvar o dicionário com todos os vectorizers (opcional, mas pode ser útil)
# all_tfidf_path = os.path.join(OUTPUT_DIR, "all_tfidf_vectorizers.pkl")
# joblib.dump(tfidf_vectorizers, all_tfidf_path)
# print(f"\nDicionário de todos TFIDF Vectorizers salvo em: {all_tfidf_path}")

Disciplinas encontradas: ['Paisagismo' 'Arquitetura' 'Climatização' 'Pressurização' 'Elétrica'
 'Estrutura' 'Gás' 'Hidráulica' 'Impermeabilização' 'Interiores'
 'Luminotécnico' 'Preventivo contra incêndio' 'SPDA' 'Piscinas'
 'Sanitário' 'Telemática']

============================== PROCESSANDO DISCIPLINA: Paisagismo ==============================
-> Número de amostras para 'Paisagismo': 505
-> Features iniciais selecionadas: ['Class', 'PredefinedType', 'BuildingStorey', 'Material', 'PSET_RÔGGA.RÔGGA_SEÇÃO', 'PSET_RÔGGA.RÔGGA_DESCRIÇÃO', 'Name']
-> Aplicando TF-IDF no 'Name'...
   Coluna 'Name' substituída por 100 features TF-IDF.
   Shape atual de X: (505, 106)
-> Aplicando Frequency Encoding...
-> Aplicando One-Hot Encoding...
   Shape após OHE: (505, 115)
-> Filtrando classes com menos de 3 amostras...
   Amostras removidas: 1
-> Total de classes ÚNICAS para o modelo 'Paisagismo': 6
-> Dividindo dados em treino/teste...
   Shape X_train: (378, 115)
-> Iniciando treinamento XGBoost co

c:\Users\lucas.galicioli\Desktop\ifc-classifier\.venv\Lib\site-packages\xgboost\callback.py:386: UserWarning: [16:17:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


   Treinamento concluído! Melhor iteração: 304
-> Avaliando o modelo...
   Acurácia FINAL para 'Paisagismo': 0.9921 (99.21%)

   Relatório de Classificação Detalhado para 'Paisagismo':
                                                   precision    recall  f1-score   support

                                  Camada drenante       1.00      1.00      1.00         7
                                         Caminhos       0.50      1.00      0.67         1
                                        Canteiros       1.00      1.00      1.00         4
Elementos horizontais de concreto moldado in loco       1.00      1.00      1.00         4
                            Substrato da Floreira       1.00      1.00      1.00         4
                                     Unclassified       1.00      0.99      1.00       106

                                         accuracy                           0.99       126
                                        macro avg       0.92      1.00      0.94     

c:\Users\lucas.galicioli\Desktop\ifc-classifier\.venv\Lib\site-packages\xgboost\callback.py:386: UserWarning: [16:17:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


   Treinamento concluído! Melhor iteração: 351
-> Avaliando o modelo...
   Acurácia FINAL para 'Arquitetura': 0.9949 (99.49%)

   Relatório de Classificação Detalhado para 'Arquitetura':
                                                precision    recall  f1-score   support

                                Aba de Fachada       0.97      1.00      0.99        35
                Bancadas em Mármore ou Granito       1.00      1.00      1.00        77
                    Barra de Apoio Articulável       0.33      0.50      0.40         4
                           Barra de Apoio em L       0.93      1.00      0.97        42
                                    Bate-rodas       1.00      1.00      1.00        99
                   Churrasqueira pré-fabricada       0.92      1.00      0.96        12
                                      Corrimão       1.00      1.00      1.00         3
                                       Drywall       0.83      1.00      0.91         5
                    

...

   Treinamento concluído! Melhor iteração: 242
-> Avaliando o modelo...
   Acurácia FINAL para 'Piscinas': 1.0000 (100.00%)

   Relatório de Classificação Detalhado para 'Piscinas':
                                                              precision    recall  f1-score   support

                                        Aquecedor de piscina       1.00      1.00      1.00         3
                      Bomba de pressurização de água (AQ/AF)       1.00      1.00      1.00         8
Caixas de passagem, caixas de passagem em aduela de concreto       1.00      1.00      1.00         2
               Conexões, dispositivos de aspiração e retorno       1.00      1.00      1.00       266
                                           Dosador de Ozônio       1.00      1.00      1.00         5
                                                      Filtro       1.00      1.00      1.00         2
                                                       Ralos       1.00      1.00      1.00         1
 

c:\Users\lucas.galicioli\Desktop\ifc-classifier\.venv\Lib\site-packages\xgboost\callback.py:386: UserWarning: [16:18:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


   Treinamento concluído! Melhor iteração: 235
-> Avaliando o modelo...
   Acurácia FINAL para 'Sanitário': 0.9463 (94.63%)

   Relatório de Classificação Detalhado para 'Sanitário':
                                                              precision    recall  f1-score   support

                     Bombas de recalque, de drenagem pluvial       1.00      1.00      1.00         1
Caixas de passagem, caixas de passagem em aduela de concreto       1.00      1.00      1.00        23
                                                    Conexões       1.00      1.00      1.00      3571
                                                       Ralos       0.81      1.00      0.89        51
                                             Ralos Sifonados       1.00      1.00      1.00       257
                                        Registros e válvulas       1.00      1.00      1.00        23
                               Tubulação de Drenagem e Reúso       0.99      0.99      0.99       798


c:\Users\lucas.galicioli\Desktop\ifc-classifier\.venv\Lib\site-packages\xgboost\callback.py:386: UserWarning: [16:18:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


   Treinamento concluído! Melhor iteração: 183
-> Avaliando o modelo...
   Acurácia FINAL para 'Telemática': 1.0000 (100.00%)

   Relatório de Classificação Detalhado para 'Telemática':
                       precision    recall  f1-score   support

    Caixa de Passagem       1.00      1.00      1.00        69
           Caixas 4X2       1.00      1.00      1.00        67
Conexão de Eletroduto       1.00      1.00      1.00       315
         Eletrocalhas       1.00      1.00      1.00       469
Eletrodutos Flexíveis       1.00      1.00      1.00       357
  Eletrodutos Rígidos       1.00      1.00      1.00        14
Placas de Sinalização       1.00      1.00      1.00         5
      Ponto de câmera       1.00      1.00      1.00        17
         Unclassified       1.00      1.00      1.00         1

             accuracy                           1.00      1314
            macro avg       1.00      1.00      1.00      1314
         weighted avg       1.00      1.00      1.00    